# 09 — Threat Intelligence and Attribution

## What This Is
Threat intelligence (TI) transforms raw security data into actionable knowledge about adversaries, their TTPs (Tactics, Techniques, Procedures), and infrastructure. Attribution connects incidents to threat actor groups.

## Why It Matters
TI-driven security moves from reactive (respond after breach) to predictive (anticipate based on known actor behaviour). MITRE ATT&CK provides a common TTP taxonomy enabling TI sharing across organisations.

## Where the Community Stands
ISACs (Information Sharing and Analysis Centers) facilitate sector-specific TI sharing. STIX/TAXII is the standard format for structured TI exchange. Commercial TI platforms (Recorded Future, Mandiant) combine open and proprietary sources.

## Authorized Testing Context
All threat actor data here is synthetic or based on publicly documented groups (from official reports like Mandiant M-Trends, CrowdStrike GTR). Attribution is probabilistic, not definitive.

In [ ]:
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Set
import hashlib, random

# MITRE ATT&CK Tactics
TACTICS = [
    'Reconnaissance', 'Resource Development', 'Initial Access',
    'Execution', 'Persistence', 'Privilege Escalation',
    'Defense Evasion', 'Credential Access', 'Discovery',
    'Lateral Movement', 'Collection', 'Command & Control',
    'Exfiltration', 'Impact'
]

# Synthetic threat actor profiles
THREAT_ACTORS = {
    'BEAR-29': {
        'aliases':    ['Cozy Bear', 'APT29'],
        'nation':     'Russia (attributed)',
        'motivation': 'Espionage',
        'ttps': [
            'T1566.001',  # Spear Phishing Attachment
            'T1078',      # Valid Accounts
            'T1053',      # Scheduled Task/Job
            'T1027',      # Obfuscated Files
            'T1041',      # Exfiltration over C2
        ],
        'infrastructure': ['fast-flux DNS', 'compromised websites', 'cloud storage'],
        'targets': ['Government', 'Defense', 'Think Tanks'],
    },
    'SPIDER-19': {
        'aliases':    ['Carbon Spider', 'FIN7'],
        'nation':     'Eastern Europe (attributed)',
        'motivation': 'Financial',
        'ttps': [
            'T1566.001',  # Spear Phishing
            'T1055',      # Process Injection
            'T1071',      # App Layer Protocol C2
            'T1486',      # Data Encrypted (ransomware)
            'T1003',      # OS Credential Dumping
        ],
        'infrastructure': ['bulletproof hosting', 'stolen VPS accounts'],
        'targets': ['Retail', 'Hospitality', 'Financial'],
    },
    'PANDA-40': {
        'aliases':    ['Deep Panda', 'APT41'],
        'nation':     'China (attributed)',
        'motivation': 'Espionage + Financial',
        'ttps': [
            'T1190',      # Exploit Public-Facing Application
            'T1059',      # Command and Scripting Interpreter
            'T1021',      # Remote Services (lateral movement)
            'T1560',      # Archive Collected Data
            'T1048',      # Exfiltration over Alt Protocol
        ],
        'infrastructure': ['cloud providers', 'tor', 'domain fronting'],
        'targets': ['Healthcare', 'Telecom', 'Semiconductor'],
    },
}

for actor_id, profile in THREAT_ACTORS.items():
    print(f'{actor_id} ({profile["aliases"][0]}) — {profile["motivation"]}: {len(profile["ttps"])} TTPs')

In [ ]:
@dataclass
class Incident:
    id: str
    ttps_observed: List[str]
    iocs: Dict[str, List[str]]
    targets: List[str]
    malware_families: List[str]

def attribute_incident(incident: Incident, actors: Dict) -> List[Dict]:
    """Score incident against known threat actor profiles."""
    scores = []
    obs_set = set(incident.ttps_observed)
    obs_targets = set(incident.targets)

    for actor_id, profile in actors.items():
        actor_ttps    = set(profile['ttps'])
        actor_targets = set(profile['targets'])

        ttp_overlap    = len(obs_set & actor_ttps)
        target_overlap = len(obs_targets & actor_targets)

        # Jaccard similarity for TTPs
        union = len(obs_set | actor_ttps)
        jaccard = ttp_overlap / union if union > 0 else 0.0

        confidence = jaccard * 0.6 + (target_overlap / max(1,len(actor_targets))) * 0.4
        confidence = round(min(1.0, confidence), 3)

        scores.append({
            'actor':           actor_id,
            'alias':           profile['aliases'][0],
            'motivation':      profile['motivation'],
            'ttp_overlap':     ttp_overlap,
            'target_overlap':  target_overlap,
            'confidence':      confidence,
            'label':           'HIGH' if confidence > 0.5 else ('MEDIUM' if confidence > 0.25 else 'LOW')
        })

    return sorted(scores, key=lambda x: x['confidence'], reverse=True)

# Simulated incident
incident = Incident(
    id='INC-2024-0042',
    ttps_observed=['T1566.001', 'T1078', 'T1027', 'T1041', 'T1003'],
    iocs={'ip': ['203.0.113.5', '198.51.100.10'], 'domain': ['c2.sync-cloud.net']},
    targets=['Government', 'Think Tanks'],
    malware_families=['SUNBURST-like', 'TEARDROP-like']
)

attributions = attribute_incident(incident, THREAT_ACTORS)
print(f'Attribution analysis for {incident.id}:')
print(f'  Observed TTPs: {incident.ttps_observed}')
print(f'  Targets:       {incident.targets}\n')
for a in attributions:
    print(f'  [{a["label"]}] {a["actor"]} ({a["alias"]})')
    print(f'    Confidence: {a["confidence"]}  TTP overlap: {a["ttp_overlap"]}  Target overlap: {a["target_overlap"]}')

## Diamond Model of Intrusion Analysis
The Diamond Model structures intrusions across four axes: Adversary, Capability, Infrastructure, Victim. Pivoting across edges reveals broader campaign patterns.

In [ ]:
@dataclass
class DiamondEvent:
    adversary:      str
    capability:     str   # malware/tool used
    infrastructure: str   # C2/delivery channel
    victim:         str
    timestamp:      str
    confidence:     float

class DiamondCampaignGraph:
    def __init__(self):
        self.events: List[DiamondEvent] = []

    def add(self, event: DiamondEvent):
        self.events.append(event)

    def pivot_by_infrastructure(self, infra: str) -> List[DiamondEvent]:
        """Find all events sharing same infrastructure (C2 pivot)."""
        return [e for e in self.events if e.infrastructure == infra]

    def pivot_by_capability(self, cap: str) -> List[DiamondEvent]:
        """Find all events using same malware/tool."""
        return [e for e in self.events if e.capability == cap]

    def cluster_campaign(self) -> Dict[str, List[DiamondEvent]]:
        """Cluster events by adversary to identify campaigns."""
        campaigns = {}
        for e in self.events:
            campaigns.setdefault(e.adversary, []).append(e)
        return campaigns

graph = DiamondCampaignGraph()
graph.add(DiamondEvent('BEAR-29', 'SUNBURST-variant', 'c2.sync-cloud.net', 'US-GOV-001', '2024-01-15', 0.85))
graph.add(DiamondEvent('BEAR-29', 'TEARDROP',         'c2.sync-cloud.net', 'EU-THINK-002','2024-01-20', 0.75))
graph.add(DiamondEvent('SPIDER-19','DARKSIDE',         'bulletproof.ru',   'RETAIL-003',  '2024-02-01', 0.90))
graph.add(DiamondEvent('BEAR-29', 'SUNBURST-variant', 'fast-flux-001.net', 'US-GOV-004',  '2024-02-10', 0.80))

# Pivot: find all events using same C2
c2_pivot = graph.pivot_by_infrastructure('c2.sync-cloud.net')
print(f'Pivot on c2.sync-cloud.net: {len(c2_pivot)} events')
for e in c2_pivot:
    print(f'  {e.timestamp} | {e.adversary} | {e.victim}')

# Campaign clustering
campaigns = graph.cluster_campaign()
print(f'\nCampaigns identified:')
for actor, events in campaigns.items():
    victims = [e.victim for e in events]
    print(f'  {actor}: {len(events)} events targeting {victims}')

## TTP Coverage Analysis
Security controls are mapped to ATT&CK techniques to find coverage gaps. This is the foundation of a defence-in-depth strategy.

In [ ]:
SECURITY_CONTROLS = {
    'Email Gateway':         ['T1566.001', 'T1566.002'],
    'EDR':                   ['T1055', 'T1059', 'T1003', 'T1027'],
    'Network IDS':           ['T1071', 'T1041', 'T1048'],
    'SIEM/Log Monitoring':   ['T1078', 'T1021'],
    'Vuln Mgmt':             ['T1190'],
    'Backup/Recovery':       ['T1486'],
}

def coverage_analysis(actors: Dict, controls: Dict) -> None:
    all_ttps = set()
    for profile in actors.values():
        all_ttps |= set(profile['ttps'])

    covered = set()
    for ctrl_ttps in controls.values():
        covered |= set(ctrl_ttps)

    gaps = all_ttps - covered
    pct  = len(covered & all_ttps) / len(all_ttps) * 100

    print(f'Total unique TTPs (across all actors): {len(all_ttps)}')
    print(f'Covered by controls:                   {len(covered & all_ttps)} ({pct:.0f}%)')
    print(f'Gaps (no control):                     {len(gaps)}')
    if gaps:
        print(f'  Uncovered TTPs: {sorted(gaps)}')
    print('\nControl coverage:')
    for ctrl, ttps in controls.items():
        actor_ttps_covered = len(set(ttps) & all_ttps)
        print(f'  {ctrl:<25} covers {actor_ttps_covered} relevant TTPs')

coverage_analysis(THREAT_ACTORS, SECURITY_CONTROLS)